# Plot sfc_mean2t (HEALPix) — January 1990 (Robinson)

This notebook reproduces the Robinson-projection plot (Cartopy) for the HEALPix **NESTED** field `mean2t` from:
- `/work/ab0995/a270135/MN5/projt319/a3be_netcdf/sfc_mean2t/sfc_mean2t_1990.nc`

## Important: environment on Levante
Run with a Python environment that has `cartopy`. We used:
- `module load esmvaltool/2.11.0`

(We also installed `astropy` + `astropy-healpix` with `pip --user`.)


In [ ]:
import glob
import os
import re

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from astropy_healpix import HEALPix
import astropy.units as u


In [ ]:
# Input/output paths (map plot)
path_nc = '/work/ab0995/a270135/MN5/projt319/a3be_netcdf/sfc_mean2t/sfc_mean2t_1990.nc'
out_png = '/work/ab0995/a270135/MN5/projt319/a3be_netcdf/sfc_mean2t_figures/sfc_mean2t_1990_Jan_robinson_degC_-35_35.png'


In [ ]:
# Load January (time index 0)
ds = xr.open_dataset(path_nc)
vals_hp = ds['mean2t'].isel(time=0, height=0).values.astype('float64')
npix = vals_hp.size
nside = int(round(np.sqrt(npix / 12.0)))
assert 12 * nside * nside == npix, f'npix={npix} is not 12*nside^2'
print('npix:', npix, 'nside:', nside)


In [ ]:
# Regrid HEALPix (NESTED) -> regular 1° lat/lon grid (nearest-neighbor sampling)
lon = np.arange(-180.0, 180.0, 1.0)
lat = np.arange(-90.0, 90.0 + 1.0, 1.0)
lon2d, lat2d = np.meshgrid(lon, lat)

hp = HEALPix(nside=nside, order='nested', frame=None)
idx = hp.lonlat_to_healpix(lon2d * u.deg, lat2d * u.deg)
grid_vals = vals_hp[idx]
print('grid shape:', grid_vals.shape)


In [ ]:
# Plot (Robinson projection)
fig = plt.figure(figsize=(13, 6))
ax = plt.axes(projection=ccrs.Robinson())

# Convert to °C for plotting
plot_vals = grid_vals - 273.15

# Discrete 3°C color bands
levels = np.arange(-35, 36, 3)
norm = plt.matplotlib.colors.BoundaryNorm(levels, ncolors=plt.get_cmap('RdBu_r').N, clip=True)

im = ax.pcolormesh(
    lon2d, lat2d, plot_vals,
    transform=ccrs.PlateCarree(),
    cmap='RdBu_r',
    norm=norm,
    shading='auto'
)

ax.coastlines(linewidth=0.7)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, color='gray', alpha=0.2)
ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.5)

cb = plt.colorbar(im, orientation='horizontal', pad=0.05, fraction=0.05)
cb.set_label('mean2t (°C)', fontsize=10)
cb.set_ticks(np.arange(-35, 36, 3))

plt.title('mean2t (2m air temperature) — January 1990 (HEALPix NESTED → 1° grid)', fontsize=12)
plt.tight_layout()
plt.savefig(out_png, dpi=200)
plt.show()
print('saved:', out_png)


## Global mean 2m temperature (annual mean per year) + trend

We compute one global annual-mean value per available yearly NetCDF file in `sfc_mean2t/`.

**Why simple mean works here:** HEALPix pixels have equal area, so `mean(gsize)` is already area-weighted.


In [ ]:
# Discover available yearly files
base_dir = '/work/ab0995/a270135/MN5/projt319/a3be_netcdf/sfc_mean2t'
pattern = os.path.join(base_dir, 'sfc_mean2t_*.nc')
files = sorted(glob.glob(pattern))
print('found', len(files), 'files')
files


In [ ]:
# Compute global annual mean for each year
years = []
global_means_k = []

for f in files:
    m = re.search(r'sfc_mean2t_(\d{4})\.nc$', f)
    if not m:
        continue
    year = int(m.group(1))

    ds_y = xr.open_dataset(f)
    da = ds_y['mean2t'].isel(height=0)

    # average over months and pixels
    gm = da.mean(dim=('time', 'gsize'), skipna=True).item()

    years.append(year)
    global_means_k.append(gm)

order = np.argsort(years)
years = np.array(years)[order]
global_means_k = np.array(global_means_k)[order]

print('years:', years)
print('global mean (K):', global_means_k)


In [ ]:
# Plot time series + linear trend
global_means_c = global_means_k - 273.15

slope_c_per_year, intercept = np.polyfit(years, global_means_c, 1)
trend = slope_c_per_year * years + intercept

plt.figure(figsize=(10, 4))
plt.plot(years, global_means_c, marker='o', linewidth=2, label='Global mean')
plt.plot(years, trend, 'k--', linewidth=2, label=f'Trend = {slope_c_per_year:.3f} °C/year')

plt.title('Global mean 2m air temperature (annual mean)')
plt.xlabel('Year')
plt.ylabel('Temperature (°C)')
plt.grid(True, alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

print(f'Trend: {slope_c_per_year*10:.3f} °C/decade')
